In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("IMDb MoviesIndia.csv",encoding='latin1')
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [3]:
df.dropna(subset=["Rating"], inplace=True)
df["Duration"] = df["Duration"].astype(str).str.replace(' min', '', regex=False)
df["Duration"] = pd.to_numeric(df["Duration"], errors='coerce')

In [4]:
df["Duration"] = df["Duration"].fillna(df["Duration"].median())
df["Votes"] = df["Votes"].astype(str).str.replace(',', '')
df["Votes"] = pd.to_numeric(df["Votes"], errors='coerce')

In [5]:
df["Votes"] = df["Votes"].fillna(df["Votes"].median())

In [6]:
df["Director"] = df["Director"].fillna(df["Director"].mode()[0])
df["Actor 1"] = df["Actor 1"].fillna(df["Actor 1"].mode()[0])
df["Actor 2"] = df["Actor 2"].fillna(df["Actor 2"].mode()[0])
df["Actor 3"] = df["Actor 3"].fillna(df["Actor 3"].mode()[0])
df["Genre"] = df["Genre"].fillna("Unknown")

In [7]:
df["Main Genre"] = df["Genre"].apply(lambda x: x.split(",")[0])

In [8]:
categorical_features = ["Main Genre", "Director", "Actor 1"]
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_features = encoder.fit_transform(df[categorical_features])
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out())

In [9]:
numerical = df[["Duration", "Votes"]].reset_index(drop=True)
X = pd.concat([numerical, encoded_df], axis=1)
y = df["Rating"].reset_index(drop=True)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
best_year = df.groupby("Year")["Rating"].mean().idxmax()
best_rating = df.groupby("Year")["Rating"].mean().max()
print("Best Year:", best_year, "with Rating:", round(best_rating, 2))

In [ ]:
sns.scatterplot(x=df["Duration"], y=df["Rating"])
plt.title("Duration vs Rating")
plt.show()

In [ ]:
top10_overall = df.sort_values("Rating", ascending=False)[["Name", "Rating"]].head(10)
print("Top 10 Movies Overall:")
print(top10_overall)

In [ ]:
top10_per_year = (
    df.sort_values("Rating", ascending=False)
    .groupby("Year", group_keys=False)  
    .head(10)  
)

In [ ]:
top10_per_year.reset_index(drop=True, inplace=True)
print("Top 10 Movies per Year:")
print(top10_per_year[["Year", "Name", "Rating"]])

In [ ]:
popular_movies_year = df[df["Votes"] > df["Votes"].median()].groupby("Year").size()
popular_movies_year.plot(kind="bar", figsize=(10,5))
plt.title("Number of Popular Movies per Year")
plt.show()

In [ ]:
top_voted = df.sort_values(["Votes", "Rating"], ascending=False)[["Name", "Votes", "Rating"]].head(10)
print("Top Voted Movies with High Ratings:")
print(top_voted)

In [ ]:
most_movies_director = df["Director"].value_counts().idxmax()
director_count = df["Director"].value_counts().max()
print(f"Most Active Director: {most_movies_director} with {director_count} movies")

In [ ]:
actors = pd.concat([df["Actor 1"], df["Actor 2"], df["Actor 3"]])
most_common_actor = actors.value_counts().idxmax()
actor_count = actors.value_counts().max()
print(f"Most Featured Actor: {most_common_actor} in {actor_count} movies")

In [ ]:
genre_ratings = df.groupby("Main Genre")["Rating"].mean()
genre_ratings.sort_values(ascending=False).plot(kind="bar", figsize=(10,5))
plt.title("Average Rating by Genre")
plt.xlabel("Genre")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
sns.scatterplot(x=df["Votes"], y=df["Rating"])
plt.title("Votes vs Rating")
plt.xlabel("Votes")
plt.ylabel("Rating")
plt.show()